# HistorianRCA: starter analysis notebook

A toolkit for finding **candidate** abnormal periods on TK-101. It does not give answers. Every candidate has to be diagnosed from first principles using `PROCESS_DESCRIPTION.md`.

**Workflow:**
1. Load the data.
2. Map the data-quality artifacts and operator actions. These are distractors, not faults.
3. Compute physics and redundancy residuals. With healthy equipment each one sits near a constant.
4. Score the residuals against the first 14 days (normal reference).
5. Merge exceedances into candidate windows.
6. Drill into each window on the **primary** (compressed) record.
7. Write down your own conclusions and export the submission CSV.

## 1. Setup and load

**What this cell does:** Loads the 1-minute grid, the primary compressed record and the tag dictionary. Defines `raw(tag)` for the stored good-quality points of one tag, and sets the 14-day baseline window.

**Output:** Data sizes, the record period and the baseline dates. Expect 129,600 grid rows (90 days × 1440 min), 90 tags, and 30 TK-101 tags.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["axes.grid"] = True

wide = pd.read_parquet("historian_60s.parquet").set_index("timestamp")
long = pd.read_parquet("historian.parquet")
tags = pd.read_csv("tags.csv").set_index("tag")

tk_tags = tags.index[tags.unit == "TK-101"].tolist()
t0 = wide.index[0]
BASELINE = (t0, t0 + pd.Timedelta(days=14))

# Primary record, split per tag once (good-quality points only)
_raw = {t: g.set_index("timestamp")["value"] for t, g in long[long.quality == 0].groupby("tag")}
def raw(tag):
    """Good-quality stored points for one tag from the primary (compressed) record."""
    return _raw[tag]

print(wide.shape, "grid rows x tags |", len(long), "stored points |", len(tk_tags), "TK-101 tags")
print("Record:", wide.index[0], "->", wide.index[-1])
print("Baseline:", BASELINE)

### Constants from the process description
All values come from sections 3–7 of `PROCESS_DESCRIPTION.md`.

**What this cell does:** Declares the plant constants, transport delays and steam/purity correlations from the process description, plus a helper that shifts a downstream tag back to vessel time.

**Output:** No output. Every later cell uses these definitions.

In [ ]:
UA_CLEAN = 72_000.0   # W/K, clean coil, TV-101 fully open (4.3)
R_TV     = 50.0       # TV-101 rangeability (4.3)
CV_LV    = 21.24      # m3/h per unit opening per m^0.5 (4.1)
UA_LOSS  = 500.0      # W/K vessel heat loss (4.2)
RHO_CP   = 1000 * 4.18  # kJ/m3.K
FIC_SPAN = 3000.0     # kg/h, FIC-201.SP = 30 x TIC-101.OP (5.1)

# Transport delays in minutes (6): shift the downstream tag back to vessel time
DELAY = {"FI-102": 2, "TI-102": 3, "TI-202": 4, "AI-101": 8 + 2}  # AI-101: 8 min sample + ~90 s lag

def t_sat(p_barg):   return 100.0 * (p_barg + 1.013) ** 0.25      # degC (4.4)
def h_fg(tsat_c):    return 2500.0 - 2.4 * tsat_c                   # kJ/kg (4.4)
def purity(T):       return 99.4 - 0.02 * (T - 78.0) ** 2           # % (1)

def at_vessel_time(tag):
    """Grid series for a tag shifted so that it lines up with vessel time."""
    return wide[tag].shift(-DELAY.get(tag, 0))

## 2. Data-quality map (artifacts, not faults)
Section 8.2 describes three kinds of artifact:
- **Communication outages**: every tag on one DCS node goes bad for the same interval.
- **Stuck readings**: a tag stops updating but still reports good quality. In the compressed record this shows up as an unusually long gap between stored points.
- **Single-sample spikes**: one point that jumps away and straight back.

The stuck detector also flags controller outputs held in MAN (`TIC-101.OP` while `TIC-101.MODE` = 0). That is expected behaviour (5.1), not a stuck transmitter.

**What this cell does:** Finds communication outages: intervals where every TK-101 tag on the same DCS node is bad (NaN) at once.

**Output:** One row per outage, with the node, start, end and length in minutes. These are artifacts, not faults. Don't report them, and ignore anything odd inside them.

In [ ]:
def runs(mask, min_len=1):
    """Contiguous True runs in a boolean Series -> DataFrame(start, end, minutes)."""
    m = mask.astype(int).values
    d = np.diff(np.r_[0, m, 0])
    starts, ends = np.where(d == 1)[0], np.where(d == -1)[0] - 1
    idx = mask.index
    df = pd.DataFrame({"start": idx[starts], "end": idx[ends]})
    df["minutes"] = (ends - starts + 1)
    return df[df.minutes >= min_len].reset_index(drop=True)

# Outages: all TK-101 tags on a node NaN at once
outages = []
for node, node_tags in tags.loc[tk_tags].groupby("dcs_node").groups.items():
    r = runs(wide[list(node_tags)].isna().all(axis=1))
    r.insert(0, "node", node)
    outages.append(r)
outages = pd.concat(outages).sort_values("start").reset_index(drop=True)
print(f"{len(outages)} TK-101 node outages")
outages

**What this cell does:** Finds suspected stuck readings: gaps between stored points far longer than that tag normally shows in the baseline. A long gap means the value stopped changing.

**Output:** One row per suspected flat period, with the tag, start, end, length and the per-tag threshold used. Stuck transmitters are artifacts. A held output during MAN mode is normal.

In [ ]:
# Stuck readings: gap between stored good points much longer than that tag's normal gap
analog = [t for t in tk_tags if tags.loc[t, "role"] not in ("MODE", "SP") and not t.startswith("CALC")]
stuck = []
for t in analog:
    s = raw(t)
    gaps = s.index.to_series().diff().dt.total_seconds() / 60
    base = gaps[s.index < BASELINE[1]]
    thr = max(30.0, 5 * base.quantile(0.999))    # minutes
    for end, g in gaps[gaps > thr].items():
        start = end - pd.Timedelta(minutes=g)
        # ignore gaps fully explained by a node outage
        node_out = outages[(outages.node == tags.loc[t, "dcs_node"]) &
                           (outages.start <= start + pd.Timedelta(minutes=5)) &
                           (outages.end >= end - pd.Timedelta(minutes=5))]
        if node_out.empty:
            stuck.append({"tag": t, "start": start, "end": end, "minutes": round(g), "threshold_min": round(thr)})
stuck = pd.DataFrame(stuck).sort_values("start").reset_index(drop=True) if stuck else pd.DataFrame(columns=["tag","start","end","minutes"])
print(f"{len(stuck)} suspected stuck/flat periods")
stuck

**What this cell does:** Finds single-sample spikes: one stored point that jumps more than 5 % of span away from both neighbours, while the neighbours agree with each other.

**Output:** The total spike count, then a table of tag-days with 5 or more spikes. Isolated spikes are artifacts. A cluster on one tag in one day is not isolated and may be real erratic behaviour worth a look.

In [ ]:
# Single-sample spikes: a point far from both neighbours while the neighbours agree
spikes = []
for t in analog:
    s = raw(t)
    if len(s) < 10:
        continue
    d_prev, d_next = s - s.shift(1), s.shift(-1) - s
    span = tags.loc[t, "range_hi"] - tags.loc[t, "range_lo"]
    k = 0.05 * span                                    # 5 % of span jump
    hit = (d_prev.abs() > k) & (d_next.abs() > k) & (np.sign(d_prev) != np.sign(d_next)) \
          & ((s.shift(1) - s.shift(-1)).abs() < 0.3 * k)
    for ts in s.index[hit.fillna(False).values]:
        spikes.append({"tag": t, "timestamp": ts, "value": s[ts]})
spikes = pd.DataFrame(spikes)
print(f"{len(spikes)} single-sample spikes")
# Isolated spikes are artifacts. Many "spikes" on one tag in one day are NOT isolated:
# they suggest real erratic behaviour (e.g. a sticking valve) and deserve a look.
if len(spikes):
    per_day = spikes.groupby(["tag", spikes.timestamp.dt.floor("D")]).size().rename("count")
    display(per_day[per_day >= 5].reset_index().rename(columns={"timestamp": "day"}))

## 3. Operator actions (distractors)
Setpoint and mode changes are operator entries (5). The unit's normal response to them is **not** a fault. A mode change can still be a clue, though: an operator may have reacted to a fault.

**What this cell does:** Extracts every operator setpoint and mode change (TIC-101, LIC-101, FIC-201) from the primary record.

**Output:** One row per action, with the time, tag, old value and new value. Modes: 0 = MAN, 1 = AUTO, 2 = CAS. The unit's normal response to these is not a fault, but an action can be an operator reacting to one.

In [ ]:
action_tags = ["TIC-101.SP", "LIC-101.SP", "TIC-101.MODE", "FIC-201.MODE", "LIC-101.MODE"]
acts = []
for t in action_tags:
    s = raw(t)
    ch = s[s.diff().abs() > 1e-6]
    for ts, v in ch.items():
        acts.append({"timestamp": ts, "tag": t, "from": s.shift(1)[ts], "to": v})
actions = pd.DataFrame(acts).sort_values("timestamp").reset_index(drop=True)
print(f"{len(actions)} operator actions")
actions

## 4. Overview of the main loops (hourly means)

**What this cell does:** Plots hourly means of the main loop tags over the whole record.

**Output:** One strip per tag. Red lines are operator actions and the green band is the baseline. Use it to eyeball step changes, slow trends and daily cycles (weather, steam header).

In [ ]:
def mark_actions(ax):
    for ts in actions.timestamp:
        ax.axvline(ts, color="crimson", alpha=0.35, lw=0.8)
    ax.axvspan(*BASELINE, color="green", alpha=0.06)

overview = ["TIC-101.PV", "TIC-101.SP", "FIC-201.PV", "TV-101.POS", "LIC-101.PV",
            "LV-101.POS", "FI-101", "FI-102", "PI-201", "TI-202", "AI-101", "TI-901"]
h = wide[overview].resample("1h").mean()
fig, axes = plt.subplots(len(overview), 1, figsize=(15, 2.0 * len(overview)), sharex=True)
for ax, t in zip(axes, overview):
    ax.plot(h.index, h[t], lw=0.8)
    ax.set_ylabel(t, rotation=0, ha="right", fontsize=8)
    mark_actions(ax)
axes[0].set_title("Hourly means — red lines: operator actions, green band: baseline")
plt.tight_layout()

## 5. Physics and redundancy residuals
Each residual is computed **directly from raw measurements**, not from the DCS `CALC-*` tags. A calculated tag inherits the error of any faulty input (7), so an independent version helps. Downstream instruments are shifted back by their transport delay first.

| Residual | Healthy value | Points to |
|---|---|---|
| `tsat_err` = TI-201 − T_sat(PI-201) | ≈ 0 | header pressure or temperature transmitter |
| `subcool` = TI-201 − TI-202 | ≈ 5 K | steam trap, condensate drainage, coil flooding |
| `tv_track` = TV-101.POS − FIC-201.OP | ≈ 0 | TV-101 positioner or valve |
| `lv_track` = LV-101.POS − LIC-101.OP | ≈ 0 | LV-101 positioner or valve |
| `tv_model` = TV-101.POS − expected position | ≈ const | coil UA (fouling), steam meter, valve trim |
| `lv_model` = FI-102 − Cv·x·√h | ≈ const | outlet restriction, FI-102, level |
| `mbal` = FI-101 − FI-102 | ≈ const | flowmeters, leak, inventory |
| `level_diff` = LIC-101.PV − LI-102 | ≈ const | level transmitters |
| `ti102_err` = TI-102 − TIC-101.PV | ≈ 0 | temperature transmitters, mixing |
| `ebal` = steam duty − process duty − heat loss | ≈ small const | steam meter, FI-101, temperatures, steam quality |
| `fic_sp_err` = FIC-201.SP − 30·TIC-101.OP | ≈ 0 (in CAS) | cascade link |
| `fic_track` = FIC-201.PV − FIC-201.SP | ≈ 0 | steam supply limits, saturated valve |
| `tic_err` / `lic_err` = PV − SP | ≈ 0 | loop not holding (saturated / manual) |
| `purity_err` = AI-101 − purity(T) | ≈ 0 | analyzer or temperature |

**What this cell does:** Computes the physics and redundancy residuals (see the table above) from raw measurements on the 1-minute grid, then summarises them over the baseline.

**Output:** Baseline mean, std, min, median and max per residual. Healthy values: `tsat_err` ≈ 0, `subcool` ≈ 5 K, `tv_track`/`lv_track` ≈ 0, others ≈ a small constant. If these don't hold, a constant or delay is wrong.

In [ ]:
W = wide
T     = W["TIC-101.PV"]
Tsat  = t_sat(W["PI-201"])
Q_st  = W["FIC-201.PV"] / 3600 * h_fg(W["TI-201"])                       # kW, steam side
Q_pr  = W["FI-101"] / 3600 * RHO_CP * (T - W["TI-100"])                    # kW, process side
Q_loss = UA_LOSS / 1000 * (T - W["TI-901"])                               # kW
x_exp = 1 + np.log((Q_st * 1000).clip(lower=1) / (UA_CLEAN * (W["TI-201"] - T))) / np.log(R_TV)
FI102 = at_vessel_time("FI-102")

res = pd.DataFrame({
    "tsat_err":   W["TI-201"] - Tsat,
    "subcool":    W["TI-201"] - at_vessel_time("TI-202"),
    "tv_track":   W["TV-101.POS"] - W["FIC-201.OP"],
    "lv_track":   W["LV-101.POS"] - W["LIC-101.OP"],
    "tv_model":   W["TV-101.POS"] - 100 * x_exp,
    "lv_model":   FI102 - CV_LV * W["LV-101.POS"] / 100 * np.sqrt(W["LIC-101.PV"].clip(lower=0)),
    "mbal":       W["FI-101"] - FI102,
    "level_diff": W["LIC-101.PV"] - W["LI-102"],
    "ti102_err":  at_vessel_time("TI-102") - T,
    "ebal":       Q_st - Q_pr - Q_loss,
    "fic_sp_err": W["FIC-201.SP"] - FIC_SPAN / 100 * W["TIC-101.OP"],
    "fic_track":  W["FIC-201.PV"] - W["FIC-201.SP"],
    "tic_err":    T - W["TIC-101.SP"],
    "lic_err":    W["LIC-101.PV"] - W["LIC-101.SP"],
    "purity_err": at_vessel_time("AI-101") - purity(T),
})
# Inputs of each residual (used for artifact masking)
RES_INPUTS = {
    "tsat_err": ["TI-201", "PI-201"], "subcool": ["TI-201", "TI-202"],
    "tv_track": ["TV-101.POS", "FIC-201.OP"], "lv_track": ["LV-101.POS", "LIC-101.OP"],
    "tv_model": ["TV-101.POS", "FIC-201.PV", "TI-201", "TIC-101.PV"],
    "lv_model": ["FI-102", "LV-101.POS", "LIC-101.PV"], "mbal": ["FI-101", "FI-102"],
    "level_diff": ["LIC-101.PV", "LI-102"], "ti102_err": ["TI-102", "TIC-101.PV"],
    "ebal": ["FIC-201.PV", "TI-201", "FI-101", "TIC-101.PV", "TI-100", "TI-901"],
    "fic_sp_err": ["FIC-201.SP", "TIC-101.OP"], "fic_track": ["FIC-201.PV", "FIC-201.SP"],
    "tic_err": ["TIC-101.PV"], "lic_err": ["LIC-101.PV"], "purity_err": ["AI-101", "TIC-101.PV"],
}
base = res.loc[BASELINE[0]:BASELINE[1]]
base.describe().T[["mean", "std", "min", "50%", "max"]].round(3)

**Sanity check:** in the baseline, `tsat_err` and `ti102_err` should be close to 0, `subcool` close to 5 K, and `ebal` small. If these don't hold, the model constants or delays are wrong, not the plant.

## 6. Baseline scoring
For each residual:
1. Smooth it with a 30-minute rolling median, which removes flow noise and compression error.
2. For drift-prone residuals (`DRIFT_PRONE`), remove the slow trend.
3. Mask out artifacts on its input tags (stuck readings, outages) and the period just after an operator action.
4. Convert it to a robust z-score using the baseline median and MAD.

Tunables are at the top of the cell.

**What this cell does:** Smooths each residual, removes slow drift on drift-prone ones, masks artifacts and post-action transients, then converts each to a robust z-score against the baseline.

**Output:** Per residual: the baseline median, the robust sigma (the unit of z), and the fraction of post-baseline minutes with |z| > 5. A high fraction means the residual spends a lot of time abnormal.

In [ ]:
SMOOTH       = "30min"
# Residuals whose instruments have documented slow calibration drift (8.1): score them against a
# 3-day rolling median so drift (a distractor) is removed. Slow faults on these can be hidden -- set [] to disable.
DRIFT_PRONE  = ["purity_err", "level_diff"]
DRIFT_WINDOW = "3D"
ACTION_MASK  = pd.Timedelta("2h")     # transient after an operator action
STUCK_PAD    = pd.Timedelta("10min")

smooth = res.rolling(SMOOTH, min_periods=10).median()

def interval_mask(index, intervals, pad=pd.Timedelta(0)):
    m = pd.Series(False, index=index)
    for s, e in intervals:
        m.loc[s - pad:e + pad] = True
    return m

action_mask = interval_mask(smooth.index, [(t, t + ACTION_MASK) for t in actions.timestamp])
masked = smooth.copy()
for r, inputs in RES_INPUTS.items():
    iv = stuck[stuck.tag.isin(inputs)][["start", "end"]].itertuples(index=False) if len(stuck) else []
    m = interval_mask(smooth.index, list(iv), STUCK_PAD) | action_mask
    masked.loc[m, r] = np.nan

for r in DRIFT_PRONE:
    masked[r] = masked[r] - masked[r].rolling(DRIFT_WINDOW, min_periods=600).median()

b = masked.loc[BASELINE[0]:BASELINE[1]]
center = b.median()
scale  = 1.4826 * (b - center).abs().median()
scale  = scale.where(scale > 1e-6, b.std()).replace(0, np.nan)
z = (masked - center) / scale
pd.DataFrame({"baseline_median": center, "robust_sigma": scale,
              "frac_|z|>5_after_baseline": (z.loc[BASELINE[1]:].abs() > 5).mean()}).round(4)

**What this cell does:** Draws a heatmap of the hourly maximum |z| for every residual across the 90 days.

**Output:** Dark cells are abnormal hours. Look for blocks: several residuals going dark together show which part of the unit is affected. Cyan lines are operator actions.

In [ ]:
# Heatmap: hourly max |z| per residual (clipped at 20)
zh = z.abs().resample("1h").max().clip(upper=20)
fig, ax = plt.subplots(figsize=(15, 6))
im = ax.imshow(zh.T.values, aspect="auto", cmap="magma_r", interpolation="nearest",
               extent=[mdates.date2num(zh.index[0]), mdates.date2num(zh.index[-1]), len(zh.columns) - 0.5, -0.5])
ax.set_yticks(range(len(zh.columns)), zh.columns)
ax.xaxis_date(); ax.grid(False)
for ts in actions.timestamp:
    ax.axvline(mdates.date2num(ts), color="cyan", alpha=0.4, lw=0.6)
plt.colorbar(im, ax=ax, label="hourly max |z|")
ax.set_title("Residual anomaly heatmap (cyan: operator actions)")
plt.tight_layout()

## 7. Candidate windows
A candidate is a stretch where any residual stays above `Z_THR` for at least `MIN_DUR`. Stretches less than `MERGE_GAP` apart are merged. Faults can develop slowly (CHALLENGE.md), so the true start often comes **before** the first exceedance. Look back from each candidate's start.

**What this cell does:** Thresholds the z-scores, keeps exceedances lasting at least `MIN_DUR`, and merges those within `MERGE_GAP` of each other into candidate windows.

**Output:** One row per candidate, with start, end, hours, the residuals that fired (most first), the peak |z|, and whether it falls inside the baseline. These are leads to investigate, not confirmed faults. The true start of a slow fault is usually earlier.

In [ ]:
Z_THR, MIN_DUR, MERGE_GAP = 6.0, pd.Timedelta("45min"), pd.Timedelta("3h")

flag = z.abs() > Z_THR
any_flag = flag.any(axis=1)
c = runs(any_flag)
c = c[c.minutes >= MIN_DUR / pd.Timedelta("1min")]

merged = []
for row in c.itertuples():
    if merged and row.start - merged[-1]["end"] <= MERGE_GAP:
        merged[-1]["end"] = row.end
    else:
        merged.append({"start": row.start, "end": row.end})
cand = pd.DataFrame(merged)
if len(cand):
    cand["hours"] = ((cand.end - cand.start) / pd.Timedelta("1h")).round(1)
    cand["residuals_fired"] = [", ".join(flag.loc[s:e].sum().loc[lambda x: x > 15].sort_values(ascending=False).index)
                               for s, e in zip(cand.start, cand.end)]
    cand["peak_|z|"] = [round(float(z.loc[s:e].abs().max().max()), 1) for s, e in zip(cand.start, cand.end)]
    cand["in_baseline"] = cand.start < BASELINE[1]
print(f"{len(cand)} candidate windows (not all are faults; check each one)")
cand

## 8. Drill-down
`inspect(start, end, tags)` plots the **primary** compressed points (dots) on top of the 1-minute grid (line) for a window. Operator actions are shown as red lines and outages as grey bands. Use it to confirm a candidate isn't an interpolation or compression artifact, and to work out which item failed and how.

**What this cell does:** Defines `inspect()`, which plots raw stored points (dots) over the grid (line) plus chosen residuals for a window, then runs it on the first candidate as an example.

**Output:** Stacked strips for the window ± 6 h. The gold band is the candidate, red lines are operator actions and grey bands are outages. Orange is the smoothed residual. Change the arguments to look at any candidate.

In [ ]:
def inspect(start, end, tag_list, pad="6h", residuals=None):
    s, e = pd.Timestamp(start) - pd.Timedelta(pad), pd.Timestamp(end) + pd.Timedelta(pad)
    residuals = residuals or []
    n = len(tag_list) + len(residuals)
    fig, axes = plt.subplots(n, 1, figsize=(15, 2.0 * n), sharex=True, squeeze=False)
    axes = axes[:, 0]
    for ax, t in zip(axes, tag_list):
        ax.plot(wide.loc[s:e, t], lw=0.7, color="tab:blue")
        r = raw(t).loc[s:e]
        ax.plot(r.index, r.values, ".", ms=2, color="k")
        ax.set_ylabel(t, rotation=0, ha="right", fontsize=8)
    for ax, r in zip(axes[len(tag_list):], residuals):
        ax.plot(res.loc[s:e, r], lw=0.5, color="0.6")
        ax.plot(smooth.loc[s:e, r], lw=1.0, color="tab:orange")
        ax.set_ylabel(r, rotation=0, ha="right", fontsize=8)
    for ax in axes:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), color="gold", alpha=0.12)
        for ts in actions.timestamp[(actions.timestamp >= s) & (actions.timestamp <= e)]:
            ax.axvline(ts, color="crimson", lw=0.8)
        for o in outages[(outages.end >= s) & (outages.start <= e)].itertuples():
            ax.axvspan(o.start, o.end, color="0.5", alpha=0.25)
    axes[0].set_title(f"{start} → {end}")
    plt.tight_layout()

KEY = ["TIC-101.PV", "FIC-201.PV", "TV-101.POS", "PI-201", "TI-202", "LIC-101.PV", "LI-102", "LV-101.POS", "FI-101", "FI-102"]
if len(cand):
    first = cand.iloc[0]
    inspect(first.start, first.end, KEY, residuals=first.residuals_fired.split(", ")[:3])

### Diagnosis checklist for each candidate
- Which residuals moved, and which stayed normal? Normal residuals rule causes out.
- Steam side: did TV-101 have to open further for the same steam flow? That could mean less driving force or UA, or a meter error. Did `PI-201` or `TI-201` move? What did condensate subcooling do?
- Liquid side: does `FI-101 − FI-102` persist? Do both level transmitters agree? Does `FI-102` match `Cv·x·√h`?
- Valve readback vs demand: positioner, sticking, or a valve that no longer tracks its signal.
- Is it an operator action the unit responded to normally, an artifact, weather, or calibration drift? If so, it is **not** a fault.
- Gradual onset: walk the start back to where the residual first left its baseline trend.

## 9. Your conclusions → submission CSV

**What this cell does:** Holds your own conclusions in `my_events` and writes them to `my_submission.csv` in the template's column format.

**Output:** A confirmation line with the number of rows written. It is 0 until you add events, and the columns are checked against `submission_template.csv`.

In [ ]:
# Fill in from your own analysis. kind: fault | operator_action | artifact (only faults are scored)
my_events = pd.DataFrame(columns=["start_utc", "end_utc", "kind", "root_cause", "evidence_tags", "notes"])
# Example (delete):
# my_events.loc[len(my_events)] = ["2024-03-20T10:00:00Z", "2024-03-21T04:00:00Z", "fault",
#                                  "<item, by tag> <failure mode>", "TV-101.POS;FIC-201.PV", "how you ruled out alternatives"]

def write_submission(df, path="my_submission.csv"):
    tmpl_cols = pd.read_csv("submission_template.csv").columns.tolist()
    assert list(df.columns) == tmpl_cols, f"columns must be {tmpl_cols}"
    df.to_csv(path, index=False)
    print(f"wrote {len(df)} rows to {path}")

write_submission(my_events)